# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

## Part 1: Setup & Data Loading

In [ ]:
!pip install --quiet datasets evaluate transformers[sentencepiece]

In [ ]:
import pandas as pd
from datasets import Dataset
from datasets import load_dataset

In [ ]:
# Load the UCI SMS Spam dataset from Hugging Face hub
df = pd.read_parquet("hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet")

# Convert df to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Split the data - use 4,000 samples for training and 1,000 for validation
train_ds = hf_dataset.select(range(4000))
val_ds   = hf_dataset.select(range(4000, 5000))

# Display the first few rows to understand the data structure
df.head()

In [ ]:
print(f"Training samples  : {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution (train):")
print(pd.Series(train_ds["label"]).value_counts())

## Part 2: Tokenization Setup

In [ ]:
from transformers import GPT2Tokenizer

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 has no pad token by default--set it to eos
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    return tokenizer(
        examples["sms"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

# Apply tokenization to both datasets
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok   = val_ds.map(tokenize_fn, batched=True)

print("Tokenization complete.")
print(f"Example input_ids length: {len(train_tok[0]['input_ids'])}")

## Part 3: Pre-trained Model Setup

In [ ]:
import torch
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=2,
    pad_token_id=tokenizer.eos_token_id
)

print("GPT-2 sequence classification model loaded.")
print(f"Number of labels: {model.config.num_labels}")

## Part 4: Custom Attention Implementation

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # Scaling factor prevents dot products from growing too large with dimensionality
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        # Calculate attention scores using matrix multiplication
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Apply softmax to get attention weights (normalize across the key dimension)
        attn = F.softmax(scores, dim=-1)

        # Apply attention weights to values
        return torch.matmul(attn, value), attn


print("Attention class defined.")

In [ ]:
class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        # Create embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Initialize attention layer
        self.attn = Attention(embed_dim)
        # Create final classification layer
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # Get embeddings from input
        embed = self.embedding(x)
        # Apply self-attention (query=key=value=embed)
        attn_output, _ = self.attn(embed, embed, embed)
        # Pool the attention output (mean over sequence dimension)
        pooled = attn_output.mean(dim=1)
        # Apply final linear layer
        return self.fc(pooled)


print("SimpleAttentionClassifier class defined.")

In [ ]:
def preprocess_for_attention(example):
    # Encode text using tokenizer with proper parameters
    tokens = tokenizer.encode(
        example["sms"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )
    return {"input_ids": tokens, "label": example["label"]}


# Apply preprocessing to both datasets
train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

print("Preprocessing for attention model complete.")
print(f"Example input_ids length: {len(train_ds_attn[0]['input_ids'])}")

In [ ]:
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item["input_ids"], dtype=torch.long),
            'label': torch.tensor(item["label"], dtype=torch.long)
        }


# Create data loaders
train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(val_ds_attn), batch_size=32)

print(f"Train loader batches: {len(train_loader)}")
print(f"Val loader batches  : {len(val_loader)}")

In [ ]:
# Setup training parameters
vocab_size = tokenizer.vocab_size
embed_dim = 64
num_classes = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print(f"Vocab size: {vocab_size}")

In [ ]:
# Initialize the model
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)

# Setup optimizer
optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3)

# Setup loss function
criterion = nn.CrossEntropyLoss()

print(attn_model)

In [ ]:
# Training loop
num_epochs = 3

attn_model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch in train_loader:
        inputs = batch['input_ids'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = attn_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs} | Average Loss: {avg_loss:.4f}")

print("\nCustom Attention model trained on SMS dataset. Sample batch loss:", loss.item())

## Part 5: Metrics & Evaluation

In [ ]:
import evaluate
import numpy as np

# Load evaluation metrics
accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall    = evaluate.load("recall")
f1        = evaluate.load("f1")


def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels)["precision"],
        "recall":    recall.compute(predictions=preds, references=labels)["recall"],
        "f1":        f1.compute(predictions=preds, references=labels)["f1"]
    }


print("Metrics loaded and compute_metrics function defined.")

In [ ]:
print("\nEvaluating GPT-2 Model...")
gpt2_preds = []
gpt2_labels = []

model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

gpt2_metrics = {
    "accuracy":  accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)["accuracy"],
    "precision": precision.compute(predictions=gpt2_preds, references=gpt2_labels)["precision"],
    "recall":    recall.compute(predictions=gpt2_preds, references=gpt2_labels)["recall"],
    "f1":        f1.compute(predictions=gpt2_preds, references=gpt2_labels)["f1"]
}

print("GPT-2 Metrics:", gpt2_metrics)

In [ ]:
print("\nEvaluating Custom Attention Model...")
attn_preds = []
attn_labels = []

attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = {
    "accuracy":  accuracy.compute(predictions=attn_preds, references=attn_labels)["accuracy"],
    "precision": precision.compute(predictions=attn_preds, references=attn_labels)["precision"],
    "recall":    recall.compute(predictions=attn_preds, references=attn_labels)["recall"],
    "f1":        f1.compute(predictions=attn_preds, references=attn_labels)["f1"]
}

print("Attention Model Metrics:", attn_metrics)

In [ ]:
import matplotlib.pyplot as plt

comparison_df = pd.DataFrame({
    "Metric": ["accuracy", "precision", "recall", "f1"],
    "GPT-2": [gpt2_metrics[m] for m in ["accuracy", "precision", "recall", "f1"]],
    "Custom Attention": [attn_metrics[m] for m in ["accuracy", "precision", "recall", "f1"]]
})

print(comparison_df.to_string(index=False))

x = np.arange(len(comparison_df))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, comparison_df["GPT-2"], width, label="GPT-2", color="steelblue")
plt.bar(x + width/2, comparison_df["Custom Attention"], width, label="Custom Attention", color="tomato")
plt.xticks(x, comparison_df["Metric"])
plt.ylabel("Score")
plt.title("GPT-2 vs Custom Attention Model — SMS Spam Classification")
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

## Part 6: Reflection Questions

**1. What are the roles of query, key, and value in the attention mechanism?**

The query represents what the current token is "looking for" — it encodes the information needed to decide which other tokens are relevant to it. The keys represent what each token in the sequence "offers" or contains, acting as a label that the query is compared against to compute a relevance (attention) score. The values hold the actual content that gets aggregated: once attention scores determine how much each token should contribute, the value vectors are combined in that weighted proportion to produce the final output for each position.

**2. Why do we use a scaling factor (e.g., 1/sqrt(d_k)) in the dot-product attention?**

As the embedding dimensionality d_k increases, the dot products between query and key vectors tend to grow larger in magnitude (since they sum over more dimensions), which pushes the softmax function into regions where its gradient is extremely small (saturated near 0 or 1). This makes gradients vanish during backpropagation and destabilizes training. Dividing by sqrt(d_k) rescales the dot products back to a more moderate range, keeping the softmax in a region with meaningful gradients and leading to more stable, effective learning.

**3. How does self-attention differ from traditional sequence models like RNNs?**

RNNs process sequences strictly step-by-step, where each token's representation depends on the previous hidden state, making computation inherently sequential and slow to parallelize. Self-attention, by contrast, computes relationships between all pairs of tokens simultaneously through matrix multiplication, allowing full parallelization across the sequence during training. For long-range dependencies, RNNs must propagate information through many sequential steps, which can cause it to degrade or vanish; self-attention directly connects any two tokens regardless of their distance in the sequence, in a single computation. This also changes training dynamics: self-attention models typically train faster on modern hardware (GPUs/TPUs) due to parallelism, at the cost of higher memory usage that scales quadratically with sequence length.

**4. Performance Analysis**

Based on the metrics computed above, the GPT-2 model — even without any fine-tuning on the SMS spam task — generally benefits from its large-scale pretraining on diverse text, giving it a richer understanding of language structure that often translates into stronger raw performance on a new classification head. The custom attention model, while much smaller and trained entirely from scratch on only 4,000 examples, demonstrates that even a minimal single-layer self-attention mechanism can learn meaningful patterns for spam detection, though it typically lags behind GPT-2 on most metrics due to its limited capacity and lack of pretraining.

**Trade-offs**: The custom attention model is dramatically smaller, faster to train, and fully interpretable in terms of its architecture, but it lacks the world knowledge and contextual depth that comes from large-scale pretraining. GPT-2 offers stronger performance with minimal additional training, but at the cost of significantly more parameters, slower inference, and higher memory requirements. To improve the custom attention model, one could add multiple attention heads (multi-head attention), stack several attention layers, increase the embedding dimension, add positional encodings (since the current implementation has no notion of token order), or pretrain the embedding layer on a larger text corpus before fine-tuning on the spam classification task.